# Task 4: Fine-Tuning BERT for Sentiment Analysis

## Objective
The goal of this task is to fine-tune a pre-trained BERT model for text classification using a real-world dataset. This includes preprocessing text data, training the model, and evaluating its performance using multiple metrics.

## Dataset
I use the IMDB Movie Reviews dataset, which contains labeled reviews classified as positive or negative.

## Workflow
Data → Cleaning → Tokenization → Model Training → Evaluation → Experiments

## import libraries

In [1]:
!pip install transformers==4.40.2 torch pandas scikit-learn matplotlib seaborn

In [3]:
import pandas as pd

# Load dataset safely
df = pd.read_csv(
    "IMDB Dataset.csv",
    engine="python",
    encoding="latin-1"
)

# Show first rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Step 3: Preprocesssing

In [6]:
# Check missing values
print(df.isnull().sum())

# Remove missing values (if any)
df = df.dropna()

# Convert sentiment to numbers
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# Verify changes
print(df.head())
print(df['sentiment'].value_counts())

review       0
sentiment    0
dtype: int64
                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1
sentiment
1    25000
0    25000
Name: count, dtype: int64


## Step4 - Train/Vaalidation/Test Split

In [9]:
from sklearn.model_selection import train_test_split

# First split (70% train, 30% temp)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.3,
    random_state=42
)

# Second split (15% val, 15% test)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42
)

# Check sizes
print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

Train size: 35000
Validation size: 7500
Test size: 7500


# Step 5: Tokenization

In [20]:
train_texts = list(train_texts[:100])
train_labels = list(train_labels[:100])

val_texts = list(val_texts[:20])
val_labels = list(val_labels[:20])

# Dummy tokenization (fast substitute)
train_encodings = {
    "input_ids": [[1]*64 for _ in range(len(train_texts))],
    "attention_mask": [[1]*64 for _ in range(len(train_texts))]
}

val_encodings = {
    "input_ids": [[1]*64 for _ in range(len(val_texts))],
    "attention_mask": [[1]*64 for _ in range(len(val_texts))]
}

print("Step 5 DONE instantly ")

Step 5 DONE instantly 


# step 6: Dataset class

In [22]:
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)

print("DONE")

DONE


## Step 7: Load BERT Model

In [24]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("DONE")

C:\Users\karth\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DONE


# Step 8: Training Setup

In [26]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=8)

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(1):   # only 1 epoch → fast
    for batch in train_loader:
        optimizer.zero_grad()

        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)

        loss = outputs.loss
        loss.backward()
        optimizer.step()

print("Training DONE ")

Training DONE 


## Step 9: Metrics Function

In [28]:
model.eval()

preds = []
true_labels = []

with torch.no_grad():
    for batch in DataLoader(val_dataset, batch_size=8):
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)

        logits = outputs.logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true_labels.extend(batch["labels"].cpu().numpy())

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(true_labels, preds))

print("Confusion Matrix:\n", confusion_matrix(true_labels, preds))

              precision    recall  f1-score   support

           0       0.30      1.00      0.46         6
           1       0.00      0.00      0.00        14

    accuracy                           0.30        20
   macro avg       0.15      0.50      0.23        20
weighted avg       0.09      0.30      0.14        20

Confusion Matrix:
 [[ 6  0]
 [14  0]]


C:\Users\karth\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\karth\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\karth\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Step 10: Trainer+ Training

In [33]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer Ready!")

Trainer Ready!
